# Édition contrôlée par masques — étape 3 : les 30 images

Même chaîne qu'à l'étape 2, mais sur 30 images :

1. **Sélection automatique** des 30 images dans COCO, avec l'objet, le prompt et le type d'édition.
2. **Aperçu** des 30 images et de leurs masques, pour vérifier avant de lancer les calculs.
3. **Boucle** : méthode A, méthode B et métriques pour chaque image, avec sauvegarde au fur et à mesure.
4. **Bilan** : les moyennes par méthode et par type d'édition.

**Avant de commencer** : *Exécution > Modifier le type d'exécution > GPU T4*.

## 0. Installation

In [ ]:
!pip install -q diffusers transformers accelerate pycocotools lpips
# torchao préinstallé sur Colab est trop ancien pour diffusers récent : on le retire (inutile ici)
!pip uninstall -y -q torchao

In [ ]:
import torch
assert torch.cuda.is_available(), "Pas de GPU : Exécution > Modifier le type d'exécution > GPU T4"
print("GPU :", torch.cuda.get_device_name(0))

## 1. Paramètres

Les mêmes réglages pour les 30 images et pour les deux méthodes, pour que la comparaison soit équitable.

In [ ]:
SEED = 42
NB_ETAPES = 30
GUIDANCE = 7.5
FORCE_CONTROLNET = 0.8
DILATATION_PX = 8        # élargissement du masque envoyé au modèle
MARGE_LPIPS_PX = 16      # bande ignorée autour du masque pour LPIPS (problème de voisinage)
TAILLE_MAX = 512
CANNY_BAS, CANNY_HAUT = 100, 200
NEGATIVE_PROMPT = "blurry, deformed, low quality, cartoon, painting"

## 2. Le plan d'expérience : 30 éditions

Pour chaque **type d'édition**, 10 objets COCO différents, chacun avec son prompt.

- **couleur_matiere** : même objet, autre couleur ou matière. La forme ne change pas → ControlNet devrait être avantagé.
- **remplacement** : l'objet devient autre chose, avec une forme différente → ControlNet devrait être gêné.
- **suppression** : l'objet doit disparaître, remplacé par le décor.

**Le texte donné à CLIP** et le **sens attendu** :
- pour couleur/matière et remplacement, CLIP compare la zone au **nouvel objet** (« a green velvet couch ») : le score doit **monter** (sens +1) ;
- pour une suppression, on ne peut pas décrire « rien ». CLIP compare donc la zone à **l'objet supprimé** (« a bottle ») : le score doit **baisser** (sens −1), puisque l'objet a disparu.

La colonne `score_instruction` = gain de CLIPScore × sens. Pour les trois types, **plus elle est haute, mieux l'instruction est suivie**.

In [ ]:
PROMPT_SUPPRESSION = "empty background, the surroundings continue naturally, nothing there"

PLAN = {
    "couleur_matiere": [
        ("couch", "a green velvet couch"),
        ("car", "a bright yellow car"),
        ("bus", "a bright blue bus"),
        ("chair", "a red plastic chair"),
        ("umbrella", "a yellow umbrella"),
        ("cup", "a shiny metal cup"),
        ("vase", "a blue glass vase"),
        ("suitcase", "a silver metallic suitcase"),
        ("truck", "a bright green truck"),
        ("teddy bear", "a pink teddy bear"),
    ],
    "remplacement": [
        ("cat", "a dog"),
        ("dog", "a cat"),
        ("horse", "a zebra"),
        ("sheep", "a pig"),
        ("cow", "a horse"),
        ("elephant", "a hippopotamus"),
        ("pizza", "a chocolate cake"),
        ("clock", "a round mirror"),
        ("laptop", "a stack of books"),
        ("bowl", "a basket of apples"),
    ],
    "suppression": [
        ("bottle", None), ("backpack", None), ("handbag", None), ("fire hydrant", None),
        ("stop sign", None), ("potted plant", None), ("bench", None), ("kite", None),
        ("motorcycle", None), ("parking meter", None),
    ],
}

## 3. Sélection automatique des images dans COCO

Pour chaque objet du plan, on cherche dans COCO (val2017) une image qui respecte trois règles :

1. l'objet occupe **10 à 30 %** de l'image : c'est une édition **locale** (si aucune image ne convient, on élargit à 5–40 %) ;
2. il y a **un seul** objet de cette catégorie dans l'image, pour que l'instruction ne soit pas ambiguë ;
3. l'image n'a pas déjà été choisie.

Le tirage utilise une graine fixe : on obtient **toujours la même liste**, ce qui rend l'expérience reproductible.

Si `data/prompts.csv` existe déjà (par exemple parce que tu l'as modifié à la main), il est réutilisé tel quel.

In [ ]:
import os, csv, json, random, zipfile, urllib.request
from io import BytesIO

import cv2
import numpy as np
import pandas as pd
import requests
from PIL import Image, ImageFilter
from pycocotools.coco import COCO

os.makedirs("coco", exist_ok=True)
CHEMIN_ANN = "coco/instances_val2017.json"
if not os.path.exists(CHEMIN_ANN):
    print("Téléchargement des annotations COCO…")
    zip_path = "coco/annotations.zip"
    urllib.request.urlretrieve(
        "http://images.cocodataset.org/annotations/annotations_trainval2017.zip", zip_path)
    with zipfile.ZipFile(zip_path) as z, z.open("annotations/instances_val2017.json") as src, \
            open(CHEMIN_ANN, "wb") as dst:
        dst.write(src.read())
    os.remove(zip_path)

coco = COCO(CHEMIN_ANN)
for d in ["data/images", "data/masks", "outputs/methode_a", "outputs/methode_b", "outputs/comparaisons"]:
    os.makedirs(d, exist_ok=True)

In [ ]:
def chercher_image(categorie, deja_pris, plages=((0.10, 0.30), (0.05, 0.40)), graine=0):
    cat_id = coco.getCatIds(catNms=[categorie])[0]
    for bas, haut in plages:
        candidats = []
        for img_id in sorted(coco.getImgIds(catIds=[cat_id])):
            if img_id in deja_pris:
                continue
            anns = coco.loadAnns(coco.getAnnIds(imgIds=img_id, catIds=[cat_id], iscrowd=None))
            if len(anns) != 1 or anns[0]["iscrowd"]:
                continue
            info = coco.loadImgs(img_id)[0]
            part = anns[0]["area"] / (info["width"] * info["height"])
            if bas <= part <= haut:
                candidats.append((img_id, anns[0]["id"], part))
        if candidats:
            return random.Random(graine).choice(candidats)
    raise ValueError(f"Aucune image trouvée pour '{categorie}'")

CHEMIN_PLAN = "data/prompts.csv"
if os.path.exists(CHEMIN_PLAN):
    plan = pd.read_csv(CHEMIN_PLAN, dtype={"id": str})
    print("Plan existant réutilisé :", CHEMIN_PLAN)
else:
    lignes, deja_pris, n = [], set(), 1
    for type_edition, items in PLAN.items():
        for categorie, cible in items:
            img_id, ann_id, part = chercher_image(categorie, deja_pris)
            deja_pris.add(img_id)
            if type_edition == "suppression":
                prompt, texte_clip, sens = PROMPT_SUPPRESSION, f"a {categorie}", -1
            else:
                prompt, texte_clip, sens = f"{cible}, photorealistic, high quality", cible, 1
            lignes.append({
                "id": f"{n:02d}", "type_edition": type_edition, "categorie": categorie,
                "coco_image_id": img_id, "coco_ann_id": ann_id, "part_image": round(part, 3),
                "prompt": prompt, "texte_clip": texte_clip, "sens": sens,
            })
            n += 1
    plan = pd.DataFrame(lignes)
    plan.to_csv(CHEMIN_PLAN, index=False)

plan

## 4. Préparer les 30 images et leurs masques

Même traitement qu'à l'étape 1 : redimensionnement (côté max 512, multiple de 8), masque COCO, dilatation de 8 pixels. On sauvegarde tout dans `data/`.

In [ ]:
def taille_cible(w, h, taille_max=TAILLE_MAX):
    s = taille_max / max(w, h)
    return int(w * s) // 8 * 8, int(h * s) // 8 * 8

def preparer(ligne):
    chemin_img = f"data/images/image_{ligne.id}.png"
    chemin_msk = f"data/masks/mask_{ligne.id}.png"
    if os.path.exists(chemin_img) and os.path.exists(chemin_msk):
        return Image.open(chemin_img).convert("RGB"), Image.open(chemin_msk).convert("L")

    info = coco.loadImgs(int(ligne.coco_image_id))[0]
    ann = coco.loadAnns(int(ligne.coco_ann_id))[0]
    brute = Image.open(BytesIO(requests.get(info["coco_url"], timeout=30).content)).convert("RGB")
    W, H = taille_cible(*brute.size)
    image = brute.resize((W, H), Image.LANCZOS)
    masque = Image.fromarray((coco.annToMask(ann) * 255).astype(np.uint8)).resize((W, H), Image.NEAREST)
    noyau = np.ones((2 * DILATATION_PX + 1,) * 2, np.uint8)
    masque = Image.fromarray(cv2.dilate(np.array(masque), noyau))
    image.save(chemin_img); masque.save(chemin_msk)
    return image, masque

donnees = {}
for ligne in plan.itertuples():
    donnees[ligne.id] = preparer(ligne)
print(len(donnees), "images prêtes")

### Aperçu : vérifie la liste avant de lancer les calculs

En rouge, la zone qui sera modifiée. Si une image ne te plaît pas (objet mal choisi, masque bizarre), tu peux modifier `data/prompts.csv` : change `coco_image_id`/`coco_ann_id` ou le prompt, supprime l'image et le masque correspondants dans `data/`, puis relance les sections 3 et 4.

In [ ]:
import matplotlib.pyplot as plt

def superposer(image, masque):
    s = np.array(image).copy()
    z = np.array(masque) > 0
    s[z] = (0.5 * s[z] + 0.5 * np.array([255, 0, 0])).astype(np.uint8)
    return s

fig, axes = plt.subplots(6, 5, figsize=(20, 25))
for ax, ligne in zip(axes.ravel(), plan.itertuples()):
    image, masque = donnees[ligne.id]
    ax.imshow(superposer(image, masque))
    cible = "supprimer" if ligne.sens == -1 else ligne.texte_clip
    ax.set_title(f"{ligne.id} · {ligne.categorie} → {cible}", fontsize=11)
    ax.axis("off")
plt.tight_layout(); plt.show()

## 5. Charger les modèles (une seule fois)

Les quatre réseaux sont chargés **avant** la boucle et réutilisés pour chaque image :
- Stable Diffusion inpainting (méthodes A et B) ;
- ControlNet Canny (méthode B) ;
- LPIPS et CLIP (les juges).

In [ ]:
from diffusers import StableDiffusionInpaintPipeline, ControlNetModel, StableDiffusionControlNetInpaintPipeline
import lpips
from transformers import CLIPModel, CLIPProcessor

pipe_a = StableDiffusionInpaintPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-inpainting", torch_dtype=torch.float16).to("cuda")
controlnet = ControlNetModel.from_pretrained("lllyasviel/control_v11p_sd15_canny", torch_dtype=torch.float16)
pipe_b = StableDiffusionControlNetInpaintPipeline.from_pipe(pipe_a, controlnet=controlnet).to("cuda")
pipe_a.set_progress_bar_config(disable=True)
pipe_b.set_progress_bar_config(disable=True)

juge_lpips = lpips.LPIPS(net="alex", spatial=True).to("cuda").eval()
clip_modele = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to("cuda").eval()
clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

## 6. Les fonctions

Ce sont les mêmes qu'à l'étape 2, regroupées. Deux nouveautés :

- **LPIPS avec marge** : on ignore une bande de 16 pixels autour du masque. On ne mesure que la zone vraiment éloignée de l'édition, ce qui règle le problème de voisinage vu sur l'image des chats.
- **Images noires** : si le filtre de sécurité du modèle renvoie une image noire, on réessaie avec une autre graine (2 essais au maximum). Le nombre d'essais est noté dans les résultats.

In [ ]:
def recoller_hors_masque(original, genere, masque, flou=4):
    m = np.array(masque)
    m_flou = np.array(masque.filter(ImageFilter.GaussianBlur(flou)))
    return Image.composite(genere.resize(original.size), original, Image.fromarray(np.minimum(m, m_flou)))

def carte_canny(image):
    bords = cv2.Canny(np.array(image), CANNY_BAS, CANNY_HAUT)
    return Image.fromarray(np.stack([bords] * 3, axis=-1))

def generer(pipe, image, masque, prompt, extra):
    W, H = image.size
    for essai in range(3):
        sortie = pipe(prompt=prompt, negative_prompt=NEGATIVE_PROMPT, image=image, mask_image=masque,
                      height=H, width=W, num_inference_steps=NB_ETAPES, guidance_scale=GUIDANCE,
                      generator=torch.Generator("cuda").manual_seed(SEED + essai), **extra).images[0]
        if np.array(sortie).mean() > 5:          # pas une image noire
            return sortie, essai + 1
    return sortie, essai + 1

def vers_tenseur(img):
    t = torch.from_numpy(np.array(img.convert("RGB"))).permute(2, 0, 1).float() / 127.5 - 1
    return t.unsqueeze(0).to("cuda")

def lpips_hors_masque(original, edite, masque, marge=MARGE_LPIPS_PX):
    with torch.no_grad():
        carte = juge_lpips(vers_tenseur(original), vers_tenseur(edite.resize(original.size)))[0, 0].cpu().numpy()
    noyau = np.ones((2 * marge + 1,) * 2, np.uint8)
    zone_a_garder = cv2.dilate(np.array(masque), noyau) == 0
    return float(carte[zone_a_garder].mean())

def recadrer_sur_masque(img, masque, marge=10):
    ys, xs = np.where(np.array(masque) > 0)
    w, h = masque.size
    boite = (max(xs.min() - marge, 0), max(ys.min() - marge, 0),
             min(xs.max() + marge, w), min(ys.max() + marge, h))
    return img.resize(masque.size).crop(boite)

def clip_score(img, texte):
    entrees = clip_proc(text=[texte], images=img, return_tensors="pt", padding=True).to("cuda")
    with torch.no_grad():
        sorties = clip_modele(**entrees)
    return max(100 * (sorties.image_embeds * sorties.text_embeds).sum().item(), 0)

## 7. La boucle sur les 30 images

Pour chaque ligne du plan : méthode A, méthode B, métriques, grille avant/après.

- Chaque résultat est **sauvegardé immédiatement** dans `outputs/metriques.csv`.
- Si Colab se déconnecte, relance simplement les sections 0 à 7 : les images déjà traitées sont **sautées**.
- Si une image plante, l'erreur est affichée et la boucle **passe à la suivante**.

Compte environ 15 à 25 minutes pour les 30 images sur un T4.

In [ ]:
import time
from tqdm.auto import tqdm

CHEMIN_METRIQUES = "outputs/metriques.csv"
deja_faits = set()
if os.path.exists(CHEMIN_METRIQUES):
    deja_faits = set(pd.read_csv(CHEMIN_METRIQUES, dtype={"id": str})["id"])

def ajouter_lignes(lignes):
    df = pd.DataFrame(lignes)
    df.to_csv(CHEMIN_METRIQUES, mode="a", header=not os.path.exists(CHEMIN_METRIQUES), index=False)

def sauver_comparaison(ligne, image, masque, canny, res_a, res_b):
    fig, axes = plt.subplots(1, 5, figsize=(25, 5.5))
    for ax, (im, titre) in zip(axes, [(image, "Original"), (superposer(image, masque), "Masque"),
                                      (canny, "Carte Canny"), (res_a, "A · inpainting"),
                                      (res_b, "B · + ControlNet")]):
        ax.imshow(im); ax.set_title(titre, fontsize=13); ax.axis("off")
    fig.suptitle(f"{ligne.id} · {ligne.type_edition} · {ligne.categorie} → « {ligne.prompt} »", fontsize=14)
    plt.tight_layout()
    plt.savefig(f"outputs/comparaisons/comparaison_{ligne.id}.png", dpi=100, bbox_inches="tight")
    plt.close(fig)

erreurs = []
for ligne in tqdm(list(plan.itertuples()), desc="Images"):
    if ligne.id in deja_faits:
        continue
    try:
        debut = time.time()
        image, masque = donnees[ligne.id]
        canny = carte_canny(image)

        brut_a, essais_a = generer(pipe_a, image, masque, ligne.prompt, {})
        brut_b, essais_b = generer(pipe_b, image, masque, ligne.prompt,
                                   {"control_image": canny, "controlnet_conditioning_scale": FORCE_CONTROLNET})

        clip_orig = clip_score(recadrer_sur_masque(image, masque), ligne.texte_clip)
        lignes = []
        for methode, brut, essais, dossier in [("A · inpainting", brut_a, essais_a, "methode_a"),
                                               ("B · + ControlNet", brut_b, essais_b, "methode_b")]:
            final = recoller_hors_masque(image, brut, masque)
            final.save(f"outputs/{dossier}/image_{ligne.id}.png")
            brut.save(f"outputs/{dossier}/image_{ligne.id}_brut.png")
            clip_final = clip_score(recadrer_sur_masque(final, masque), ligne.texte_clip)
            lignes.append({
                "id": ligne.id, "type_edition": ligne.type_edition, "categorie": ligne.categorie,
                "methode": methode, "essais": essais,
                "lpips_hors_masque": lpips_hors_masque(image, final, masque),
                "lpips_hors_masque_brut": lpips_hors_masque(image, brut, masque),
                "clipscore_masque": clip_final, "clipscore_original": clip_orig,
                "gain_clipscore": clip_final - clip_orig,
                "score_instruction": (clip_final - clip_orig) * ligne.sens,
            })
            if methode.startswith("A"):
                res_a = final
            else:
                res_b = final

        sauver_comparaison(ligne, image, masque, canny, res_a, res_b)
        ajouter_lignes(lignes)
        print(f"{ligne.id} ok ({time.time() - debut:.0f} s)")
    except Exception as e:
        erreurs.append((ligne.id, repr(e)))
        print(f"{ligne.id} ERREUR : {e!r}")

print(f"Terminé. Erreurs : {len(erreurs)}", erreurs)

## 8. Bilan

Les moyennes par méthode, globalement puis par type d'édition.

- `lpips_hors_masque_brut` : **plus bas = mieux**. Le modèle a-t-il abîmé le reste de l'image ?
- `lpips_hors_masque` : **plus bas = mieux**. Vérification après recollage (devrait être très proche de 0 grâce à la marge).
- `score_instruction` : **plus haut = mieux**. L'instruction a-t-elle été suivie ?

In [ ]:
res = pd.read_csv(CHEMIN_METRIQUES, dtype={"id": str})
colonnes = ["lpips_hors_masque", "lpips_hors_masque_brut", "score_instruction"]

print("Images traitées :", res["id"].nunique(), "/ 30")
display(res.groupby("methode")[colonnes].mean().round(4))
display(res.pivot_table(index="type_edition", columns="methode", values=colonnes, aggfunc="mean").round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col, titre in [(axes[0], "lpips_hors_masque_brut", "LPIPS hors masque (brut) — plus bas = mieux"),
                       (axes[1], "score_instruction", "Score instruction (CLIP) — plus haut = mieux")]:
    res.pivot_table(index="type_edition", columns="methode", values=col).plot.bar(ax=ax, rot=0)
    ax.set_title(titre); ax.set_xlabel(""); ax.axhline(0, color="black", lw=0.8)
plt.tight_layout()
plt.savefig("outputs/bilan_metriques.png", dpi=120)
plt.show()

### Voir quelques comparaisons

Change la liste `ids` pour afficher les images qui t'intéressent.

In [ ]:
ids = ["01", "11", "21"]
for i in ids:
    chemin = f"outputs/comparaisons/comparaison_{i}.png"
    if os.path.exists(chemin):
        plt.figure(figsize=(25, 5.5)); plt.imshow(Image.open(chemin)); plt.axis("off"); plt.show()

## 9. Récupérer les fichiers

À faire **à chaque fin de session** : Colab efface tout. Le zip `resultats.zip` est aussi ton plan B pour la démo.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("data_30", "zip", ".", "data")
shutil.make_archive("resultats", "zip", ".", "outputs")
files.download("data_30.zip")
files.download("resultats.zip")

**Prochaine étape** : la notation humaine aveugle (fiches anonymisées à partir de `outputs/methode_a` et `outputs/methode_b`).